#### Problema 2 - Enriquecimiento y Clasificación de CVEs


Objetivo:
crapee  los  CVEs  publicados  en  los  últimos  3  meses  desde  cvedetails.com.  Buscar los campos  CVE  ID,  fecha  de  publicación,  CVSS  score,  tipo  de 
vulnerabilidad, vendor y producto afectado.


In [1]:
# ==========================================
# 📦 1. Imports
# ==========================================

# Data
import pandas as pd
import numpy as np

# Utilidades
import os
import json
import re
import time
from datetime import datetime, timedelta

# NLP / Similaridad
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Web scraping 
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# LLM
import google.generativeai as genai


# ==========================================
# ⚙️ 2. Configuración
# ==========================================

# 🔐 API (se maneja por variables de entorno)
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key)

MODEL_NAME = "gemini-2.5-flash"  # modelo rápido para procesamiento en batch
model = genai.GenerativeModel(MODEL_NAME)


# ==========================================
# 📁 3. Archivos
# ==========================================

INPUT_FILE = "CVE_3_meses.csv"
OUTPUT_FILE = "cve_final.csv"
INPUT_FILE = "CVE_2_dias.csv"
OUTPUT_FILE = "cve_final_2_dias.csv"
CACHE_FILE = "semantic_cache.json"


# ==========================================
# ⚙️ 4. Parámetros
# ==========================================

BATCH_SIZE = 40
THRESHOLD = 0.9

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
# ==========================================
#  Scraping de CVEs (últimos 90 días)
# ==========================================

def crear_driver():
    """Configura un driver de Selenium optimizado para scraping."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")  # evita problemas de memoria
    options.add_argument("--memory-pressure-off")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )
    return webdriver.Chrome(options=options)


def scraping_industrial_3_meses(pagina_inicio=1, output_file=INPUT_FILE):
    """
    Realiza scraping de CVEs publicados en los últimos 90 días.
    
    Estrategia:
    - Navegación paginada
    - Guardado incremental (append)
    - Reinicio del driver cada N páginas para evitar leaks de memoria
    """

    hoy = datetime.now()
    fecha_inicio = (hoy - timedelta(days=2)).strftime("%Y-%m-%d")

    print(f"Scraping desde {fecha_inicio} hasta {hoy.strftime('%Y-%m-%d')}")

    page_num = pagina_inicio
    driver = crear_driver()

    try:
        while True:

            # Reinicio periódico del navegador
            if page_num % 15 == 0:
                print(f"Reiniciando driver en página {page_num}")
                driver.quit()
                driver = crear_driver()

            url = (
                f"https://www.cvedetails.com/vulnerability-search.php?"
                f"f=1&publishdatestart={fecha_inicio}&page={page_num}"
            )

            print(f"Procesando página {page_num}")

            try:
                driver.get(url)
                time.sleep(3)

                soup = BeautifulSoup(driver.page_source, "html.parser")
                cve_blocks = soup.find_all("div", {"data-tsvfield": "cveinfo"})

                if not cve_blocks:
                    print("Fin del scraping")
                    break

                pagina_data = []

                for block in cve_blocks:
                    try:
                        pagina_data.append({
                            "CVE_ID": block.find(attrs={"data-tsvfield": "cveId"}).get_text(strip=True),
                            "Published": block.find(attrs={"data-tsvfield": "publishDate"}).get_text(strip=True),
                            "CVSS": block.find(attrs={"data-tsvfield": "maxCvssBaseScore"}).get_text(strip=True),
                            "Summary": block.find(attrs={"data-tsvfield": "summary"}).get_text(strip=True)
                        })
                    except Exception as e:
                        continue

                df_temp = pd.DataFrame(pagina_data)

                if not os.path.isfile(output_file):
                    df_temp.to_csv(output_file, index=False)
                else:
                    df_temp.to_csv(output_file, mode="a", header=False, index=False)

                page_num += 1

            except Exception as e:
                print(f"Error en página {page_num}: {e}")
                driver.quit()
                time.sleep(3)
                driver = crear_driver()

    finally:
        driver.quit()
        print(f"Scraping finalizado. Archivo generado: {output_file}")

In [ ]:
scraping_industrial_3_meses()

# Nota:
# El scraping completo demora aproximadamente 1 hora.
# Se recomienda ejecutar esta función una sola vez y reutilizar el dataset generado.

# ==========================================
#  Carga y limpieza inicial
# ==========================================

df = pd.read_csv(INPUT_FILE)

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dataset cargado: {df.shape}")
df.head()



In [ ]:
# ==========================================
#  Modelo de embeddings
# ==========================================

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# ==========================================
#  Clasificación de vulnerabilidad
# ==========================================

prototipos = {
    "Sql Injection": "sql injection database exploit",
    "XSS": "cross site scripting javascript",
    "Overflow": "buffer overflow memory crash",
    "Memory Corruption": "use after free segmentation fault",
    "CSRF": "cross site request forgery",
    "XXE": "xml external entity",
    "SSRF": "server side request forgery",
    "Open Redirect": "url redirect phishing",
    "File Inclusion": "lfi rfi include file",
    "Directory Traversal": "path traversal ../",
    "Code Execution": "remote code execution rce",
    "Privilege Escalation": "gain root privileges",
    "Bypass": "authentication bypass",
    "Denial of Service": "dos crash exhaustion",
    "Information Leak": "data leak sensitive information",
    "Input Validation": "invalid input unsanitized"
}

proto_labels = list(prototipos.keys())
proto_embeddings = embed_model.encode(list(prototipos.values()))


def clasificar_batch(textos):
    emb = embed_model.encode(textos, batch_size=64)
    sims = cosine_similarity(emb, proto_embeddings)
    idx = np.argmax(sims, axis=1)
    return [proto_labels[i] for i in idx]



In [ ]:
# ==========================================
#  Cache semántico
# ==========================================

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache_data = json.load(f)
else:
    cache_data = []


def es_valido(res):
    invalidos = ["", "n/a", "na", "unknown", "error"]
    return (
        res.get("v", "").lower() not in invalidos and
        res.get("p", "").lower() not in invalidos
    )


def buscar_cache(textos):
    if not cache_data:
        return [None]*len(textos), embed_model.encode(textos)

    cache_embs = np.array([c["embedding"] for c in cache_data])
    emb = embed_model.encode(textos)

    sims = cosine_similarity(emb, cache_embs)

    resultados = []
    for i in range(len(textos)):
        idx = np.argmax(sims[i])
        if sims[i][idx] >= THRESHOLD:
            resultados.append(cache_data[idx]["result"])
        else:
            resultados.append(None)

    return resultados, emb

In [ ]:
# ==========================================
#  Extracción con Gemini
# ==========================================

def call_gemini(prompt):
    for _ in range(3):
        try:
            r = model.generate_content(prompt)

            txt = r.text.strip() if hasattr(r, "text") else r.candidates[0].content.parts[0].text.strip()
            txt = txt.replace("```json", "").replace("```", "").strip()

            match = re.search(r'\[.*\]', txt, re.DOTALL)
            if match:
                return json.loads(match.group(0))

        except Exception as e:
            print(f"Error Gemini: {e}")
            time.sleep(2)

    return None

In [ ]:
# ==========================================
#  Pipeline de procesamiento
# ==========================================

def procesar_batch(lista):
    resultados, embeddings = buscar_cache(lista)

    idx_ia = [i for i, r in enumerate(resultados) if r is None]

    if idx_ia:
        textos_ia = [lista[i] for i in idx_ia]

        prompt = f"""
Extract vendor and product from vulnerability descriptions.

Return ONLY JSON:
[{{"v":"","p":""}}]

Texts:
{textos_ia}
"""

        data = call_gemini(prompt)

        if not data:
            data = [{"v":"N/A","p":"N/A"}]*len(textos_ia)

        for i, idx in enumerate(idx_ia):
            resultados[idx] = data[i]

            if es_valido(data[i]):
                cache_data.append({
                    "text": lista[idx],
                    "embedding": embeddings[idx].tolist(),
                    "result": data[i]
                })

    return resultados

Sistema híbrido:
- clasificación semántica rápida con embeddings
- fallback con LLM solo cuando es necesario
- cache para minimizar costo y latencia

In [ ]:
# ==========================================
#  Pipeline principal
# ==========================================

# Nota:
# Se utiliza un enfoque híbrido:
# - Embeddings → clasificación rápida de tipo de vulnerabilidad
# - LLM → extracción de vendor/producto
# - Cache → optimización de llamadas

print("Iniciando procesamiento...")

# ------------------------------------------
#  Clasificación tipo de falla
# ------------------------------------------

print("Clasificando tipo_falla...")
df["tipo_falla"] = clasificar_batch(df["Summary"].tolist())


# ------------------------------------------
#  Extracción vendor / producto (batch)
# ------------------------------------------

print("Procesando vendor/producto...")

results = []

for i in range(0, len(df), BATCH_SIZE):
    print(f"Procesando batch {i} - {i + BATCH_SIZE}")

    batch = df.iloc[i:i+BATCH_SIZE]
    res = procesar_batch(batch["Summary"].tolist())

    results.extend(res)


# ------------------------------------------
#  Validación de resultados
# ------------------------------------------

res_df = pd.DataFrame(results)

if len(res_df) != len(df):
    raise ValueError("Mismatch entre resultados y dataset")


df["vendor"] = res_df["v"].fillna("N/A")
df["producto"] = res_df["p"].fillna("N/A")


# ------------------------------------------
#  Guardado
# ------------------------------------------

df.to_csv(OUTPUT_FILE, index=False)

with open(CACHE_FILE, "w") as f:
    json.dump(cache_data, f)

print("Proceso finalizado correctamente")

Hasta acá se tarda todo aprox 70 minutos, la mayoría del tiempo se va en las llamadas a la IA, el embedding y la similitud son relativamente rápidos. El cache ayuda a reducir el tiempo en ejecuciones posteriores.



In [ ]:
# ==========================================
# Post-procesamiento y mejora de calidad
# ==========================================

# Nota:
# Se reprocesan registros con vendor/producto inválidos
# para mejorar la calidad del dataset sin reprocesar todo.


# ------------------------------------------
# 🔍 Detección de registros inválidos
# ------------------------------------------

def es_malo(x):
    if pd.isna(x):
        return True
    x = str(x).strip().lower()
    return x in ["", "n/a", "na", "unknown", "error"]


mask_bad = df["vendor"].apply(es_malo) | df["producto"].apply(es_malo)

df_bad = df[mask_bad].copy()
df_good = df[~mask_bad].copy()

print(f"Registros a reprocesar: {len(df_bad)}")


# ------------------------------------------
#  Reprocesamiento
# ------------------------------------------

results_bad = []

for i in range(0, len(df_bad), BATCH_SIZE):
    print(f"Reprocesando batch {i} - {i + BATCH_SIZE}")

    batch = df_bad.iloc[i:i+BATCH_SIZE]
    res = procesar_batch(batch["Summary"].tolist())
    results_bad.extend(res)


res_bad_df = pd.DataFrame(results_bad)

# Validación
if len(res_bad_df) != len(df_bad):
    raise ValueError("Mismatch en reprocesamiento")


# Actualización de valores
df_bad = df_bad.reset_index(drop=True)
df_bad["vendor"] = res_bad_df["v"]
df_bad["producto"] = res_bad_df["p"]

print("Reprocesamiento finalizado")


# ------------------------------------------
#  Unión de datasets
# ------------------------------------------

df_final = pd.concat([df_good, df_bad]).sort_index()

print(f"Dataset final: {len(df_final)} registros")


# ------------------------------------------
#  Limpieza final
# ------------------------------------------

df_final["vendor"] = (
    df_final["vendor"]
    .fillna("N/A")
    .replace(["", "error"], "N/A")
)

df_final["producto"] = (
    df_final["producto"]
    .fillna("N/A")
    .replace(["", "error"], "N/A")
)


# ------------------------------------------
#  Métricas de calidad
# ------------------------------------------

total = len(df_final)

valid_vendor = (~df_final["vendor"].isin(["N/A"])).sum()
valid_producto = (~df_final["producto"].isin(["N/A"])).sum()

print(f"% Vendor válido: {valid_vendor/total:.2%}")
print(f"% Producto válido: {valid_producto/total:.2%}")


# ------------------------------------------
#  Guardado final
# ------------------------------------------

df_final.to_csv(OUTPUT_FILE, index=False)

with open(CACHE_FILE, "w") as f:
    json.dump(cache_data, f)

print("Archivo final guardado correctamente")